# Task 2. Data Quality Report

### Explainable AI Credit Risk Decision Platform : Integrating Structured Borrower Data, NLP-Driven Text Intelligence, and Macroeconomic Indicators for Transparent Lending Decisions.

#### Reason:
##### To ensure that the dataset is accurate, complete and also reliable to avoid costly errors.

In [7]:
# Import libraries
# _____________________________________________

import os
import pandas as pd
import numpy as np

from pathlib import Path

In [8]:
# Loading the dataset
# ______________________________________________________

# Path to the dataset

file_path = "data/raw/accepted_2007_to_2018Q4.csv"

# Load the dataset

df = pd.read_csv(file_path, low_memory=False,nrows=500000)

# Display the first five rows

df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Data Overview 
# _____________________________________________

REPORT_DIR = "../reports:"

try:
    
    Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)
    
except PermissionError:

    print(f"Permission denied for '{REPORT_DIR}'. Using current directory instead.")
    
    REPORT_DIR = "./reports"
    
    Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)
    
except Exception as e:
    
# Handle any other potential errors
    
    print(f"Error creating directory: {e}")
    
    REPORT_DIR = "."  # Use current directory as fallback


print("CREDIT RISK DATA QUALITY REPORT")
print("="*50)

# checks are added to ensure 'df' is defined before using it
try:
    print(f"Rows: {df.shape[0]:,}")
    
    print(f"Columns: {df.shape[1]}")
    
except NameError:
    
    print("Error: DataFrame 'df' is not defined. Please load your data first.")

Permission denied for '../reports:'. Using current directory instead.
CREDIT RISK DATA QUALITY REPORT
Rows: 500,000
Columns: 151


In [10]:
# Automated Data Dictionary
# _____________________________________________

data_dictionary = pd.DataFrame({
    
    "Column Name": df.columns,
    
    "Data Type": df.dtypes.astype(str),
    
    "Missing Count": df.isnull().sum(),
    
    "Missing %": round(df.isnull().mean()*100,2),
    
    "Unique Values": df.nunique()})

data_dictionary.to_csv(f"{REPORT_DIR}/data_dictionary.csv",index=False)

In [11]:
# Full Data Quality(DQ) report
# _____________________________________________

dq_report = data_dictionary.copy()

dq_report["Duplicate Values"] = [
    
    df[col].duplicated().sum()
    
    for col in df.columns]

dq_report["Sample Value"] = [
    
    str(df[col].dropna().iloc[0])
    
    if df[col].notnull().sum() > 0
    
    else "ALL NULL"
    
    for col in df.columns]

dq_report = dq_report.sort_values(by="Missing %",ascending=False)

In [12]:
# Risk Analytics Alerts
# _____________________________________________

print("\nDATA QUALITY ALERTS")

duplicates = df.duplicated().sum()

print(f"Duplicate Rows: {duplicates:,}")

# Missing > 40

high_missing = dq_report[dq_report["Missing %"] > 40]

print(f"Columns >40% Missing: "f"{len(high_missing)}")


DATA QUALITY ALERTS
Duplicate Rows: 0
Columns >40% Missing: 58


In [13]:
# High Cardinality 

high_cardinality = dq_report[dq_report["Unique Values"]> (len(df)*0.50)]

print(f"High Cardinality Features: "f"{len(high_cardinality)}")

High Cardinality Features: 7


In [14]:
#Constance features

constant_features = dq_report[dq_report["Unique Values"] <= 1]

print(f"Constant Features: "f"{len(constant_features)}")

Constant Features: 5


In [15]:
# Target Distribution
target_distribution = (
    df["loan_status"]
    .value_counts()
    .reset_index())

target_distribution.columns = [
    "Loan Status",
    "Count"]

target_distribution["Percentage"] = (
    target_distribution["Count"]
    / target_distribution["Count"].sum()*100)

target_distribution

,Loan Status,Count,Percentage
0,Fully Paid,312340,62.468250
1,Current,104240,20.848083
2,Charged Off,78824,15.764863
3,Late (31-120 days),2977,0.595402
4,In Grace Period,1046,0.209201
5,Late (16-30 days),567,0.113400
6,Default,4,0.000800


#### Text  Quality Assessment

In [16]:
# Text Quality Assessment
# _____________________________________________

text_length = (
    df["title"]
      .fillna("")
      .astype(str)
      .str.len())

text_summary = pd.DataFrame({
    
    "Metric":[
        "Min Length",
        "Max Length",
        "Mean Length",
        "Median Length"],
    
    "Value":[
        text_length.min(),
        text_length.max(),
        text_length.mean(),
        text_length.median()]})

text_summary

,Metric,Value
0,Min Length,0.000000
1,Max Length,39.000000
2,Mean Length,18.075024
3,Median Length,18.000000


In [17]:
# Data Quality Score

completeness_score = (100 -dq_report["Missing %"].mean())

print(f"Overall Completeness: "f"{completeness_score:.2f}%")

Overall Completeness: 64.76%


In [18]:
# Outlier Report
# _____________________________________________

def outlier_summary(df, cols):

    report = []

    for col in cols:

        q1 = df[col].quantile(0.25)

        q3 = df[col].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr

        upper = q3 + 1.5 * iqr

        outliers = (
            ((df[col] < lower) |
             (df[col] > upper))
            .sum())

        report.append([
            col,
            outliers,
            round(outliers / len(df) * 100,2)])

    return pd.DataFrame(
        report,
        columns=[
            "Feature",
            "Outliers",
            "Outlier %"])

In [19]:
# Numerical feature set
# _____________________________________________

numerical_features = [
    "loan_amnt",
    "int_rate",
    "annual_inc",
    "dti",
    "revol_util"]

outlier_report = outlier_summary(df,numerical_features)

outlier_report

,Feature,Outliers,Outlier %
0,loan_amnt,3605,0.72
1,int_rate,5014,1.00
2,annual_inc,23290,4.66
3,dti,1945,0.39
4,revol_util,37,0.01


In [20]:
# Save Reports
# _____________________________________________

dq_report.to_csv(f"{REPORT_DIR}/data_quality_report.csv",index=False)

high_missing.to_csv(f"{REPORT_DIR}/missing_values_report.csv",index=False)

high_cardinality.to_csv(f"{REPORT_DIR}/cardinality_report.csv",index=False)

target_distribution.to_csv(f"{REPORT_DIR}/target_distribution.csv",index=False)

outlier_report.to_csv(f"{REPORT_DIR}/outlier_report.csv",index=False)

In [21]:
# Winsorize at 1st and 99th percentile
# _____________________________________________

def winsorize_series(series):

    lower = series.quantile(0.01)

    upper = series.quantile(0.99)

    return series.clip(lower=lower,upper=upper)
    
for col in numerical_features:

    df[col] = winsorize_series(df[col])

# Save processed data

try:
    os.makedirs("data/processed", exist_ok=True)
    
    df.to_csv("data/processed/lendingclub_clean.csv",index=False)
    
    print("File saved successfully to data/processed/")
        
except Exception as e:
    
    print(f"Error occurred: {e}")

File saved successfully to data/processed/
